# 1a — STREAM: build the question set and the raw page checkpoint

Streams Google's Natural Questions train split and keeps every question that has a short
answer of **5 words or fewer**, together with the full raw token stream of its Wikipedia
page. Nothing is cleaned here. The output is the lossless record that notebook 2 reads.

**Stream and clean are separate steps on purpose.** The stream is a fact that costs ~37
minutes for 20,000 questions; cleaning is an opinion that gets revised. Keeping the raw
tokens means a changed cleaning rule costs 12 minutes, not another stream.

**Pages are keyed by a content hash, not by url.** This run measured **214 urls that
served more than one page content**. Under url-keying, every later question on those urls
would have been scored against the wrong tokens, silently.

**Two answer strings are kept per question**, deliberately: `answer` is NQ's own string
(`'March 18, 2018'`) for scoring and comparability, `answers_rendered` is the span as the
page reads it (`'March 18 , 2018'`) for training, because a reader can only copy what is
in its context. Measured on a 100-page trial: rendered appears verbatim 78/78, NQ's text
only 65/78.

| output | |
|---|---|
| questions | 20,000 |
| pages | 18,997 |
| examples streamed | — see the metrics file |
| `step1_checkpoint.pkl` | 1.4 GB, raw tokens |
| `step1-shards/part_000..019.pkl` | 20 × 73 MB, uploaded as they land |
| runtime | ~37 min |

Sharding every 1,000 questions and uploading as it goes means a crash or a terminated pod
costs at most 1,000 questions, not the whole stream.

Structure: imports → config → definitions (nothing executes) → orchestration.

In [ ]:
import os, sys, html, pickle, time, hashlib
sys.path.insert(0, os.getcwd())
import nqrun
from itertools import islice
from collections import defaultdict
import pyarrow as pa, pyarrow.parquet as pq

import datasets.iterable_dataset as _dsi      # datasets 3.1.0 calls torch shared memory
_dsi._maybe_share_with_torch_persistent_workers = lambda v: v   # -> "Broken pipe" on macOS

In [ ]:
REV   = 'e8103d566bef4154c2c12b17c6095ec5275840cc'   # pinned NQ revision
N     = int(os.environ.get('NQ_N_QUESTIONS', 20000)) # questions to keep (bounds the pod run)
MAXT  = 5                                            # keep short answers of 5 words or fewer
Q     = {'``': '"', "''": '"'}                       # Penn-Treebank quotes -> "
SHARD = int(os.environ.get('NQ_SHARD', 1000))        # save + upload every SHARD questions
SAMPN = 50                                           # docs in the study sample
DEV   = nqrun.IS_LOCAL                               # Mac -> develop off cached rows
N_RAW = 10                                           # raw examples to cache for dev

ART   = nqrun.art('step1-checkpoint-v2')
CKPT  = str(ART / 'step1_checkpoint.pkl')
SHDIR = str(ART / 'shards')
RAW   = str(nqrun.ROOT / 'local' / f'nq_raw_{N_RAW}.pkl')
os.makedirs(SHDIR, exist_ok=True)

# fail now, not after 40 minutes of streaming: without a token hf_push warns and uploads
# nothing, and a pod's disk does not survive the pod
if not DEV:
    from huggingface_hub import get_token
    assert os.environ.get('HF_TOKEN') or get_token(), 'no HF token - nothing would upload'
    print(f"run_id {nqrun.RUN_ID} | HF repo {nqrun.HF_REPO}")

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS
#
# doc_hash uses blake2b, not hash(): hash() is randomly seeded per process, so
# it could not be compared between a local run and the pod's. The unit-separator
# join stops ['ab','c'] hashing like ['a','bc']. is_html is excluded because the
# tags are themselves tokens, so markup differences already change the hash.
def render(toks, ish, s, e):
    """Token span [s, e) as text, HTML tokens skipped. No clamp: a bad span raises."""
    return html.unescape(' '.join(Q.get(toks[i], toks[i])
                                  for i in range(s, e) if not ish[i])).strip()

def doc_hash(toks):
    """Stable page id over the tokens alone, as a blake2b hex digest."""
    return hashlib.blake2b('\x1f'.join(toks).encode(), digest_size=16).hexdigest()

def nq_stream():
    """Open the pinned Natural Questions train split as a streaming dataset."""
    from datasets import load_dataset
    return load_dataset('google-research-datasets/natural_questions', 'default',
                        split='train', streaming=True, revision=REV)

## Orchestration

On a pod the stream runs live, bounded by `N`. On a Mac it streams `N_RAW` examples once
and caches them, so the network is touched only on a cache miss — the loop below is
identical either way.

In [ ]:
# DEV streams N_RAW examples once and caches them - the network is touched only on a cache
# miss. POD streams live, bounded by N. The loop below is identical either way.
if not DEV:
    source = nq_stream()
    print(f'streaming, keeping up to {N:,} questions')
elif os.path.exists(RAW):
    source = pickle.load(open(RAW, 'rb'))['rows']
    print(f'{len(source)} raw rows from cache {RAW}')
else:
    source = list(islice(nq_stream(), N_RAW))
    os.makedirs(os.path.dirname(RAW), exist_ok=True)
    pickle.dump({'rows': source, 'dataset_revision': REV}, open(RAW, 'wb'))
    print(f'cached {len(source)} raw rows -> {RAW}')

In [ ]:
docs, qs, n, t0 = {}, [], 0, time.time()
flushed, q_mark = set(), 0          # doc hashes already in a shard, questions already in one
for ex in source:
    n += 1

    # first annotator who marked a short answer (NQ train has only one); an annotator who
    # gave none still fills the slot, with empty lists
    sa = None
    for annot in ex['annotations']['short_answers']:
        if annot['start_token']:
            sa = annot
            break
    if sa is None:
        continue

    tok = ex['document']['tokens']
    toks, ish = tok['token'], tok['is_html']

    # one annotator can mark several spans (a cast, several years) - keep all of them
    spans, ans, rend = [], [], []
    for st, en, tx in zip(sa['start_token'], sa['end_token'], sa['text']):
        st, en = int(st), int(en)
        r = render(toks, ish, st, en)
        if r and sum(1 for i in range(st, en) if not ish[i]) <= MAXT:
            spans.append([st, en])
            ans.append(tx.strip() or r)             # NQ's string:    'March 18, 2018'
            rend.append(r)                          # as the page reads: 'March 18 , 2018'
    if not spans:
        continue

    # keyed by CONTENT: a question can only resolve to the exact tokens it was annotated
    # against, and identical pages arriving under different urls merge by themselves
    h   = doc_hash(toks)
    url = ex['document']['url'] or ''
    doc = docs.setdefault(h, {'token': toks, 'is_html': ish,
                              'title': ex['document']['title'] or '', 'urls': []})
    if url and url not in doc['urls']:
        doc['urls'].append(url)

    qs.append({'question': ex['question']['text'], 'answer': ans[0], 'answers': ans,
               'answers_rendered': rend, 'doc_hash': h, 'url': url, 'raw_spans': spans})

    # shard every SHARD questions: only what is new since the last flush, uploaded as it
    # lands, so a crash or a terminated pod costs at most SHARD questions
    if len(qs) % SHARD == 0:
        new  = {k: v for k, v in docs.items() if k not in flushed}
        flushed |= new.keys()
        name = f'part_{len(qs) // SHARD - 1:03d}.pkl'
        path = os.path.join(SHDIR, name)
        pickle.dump({'documents': new, 'questions': qs[q_mark:], 'dataset_revision': REV,
                     'max_answer_tokens': MAXT}, open(path, 'wb'))
        nqrun.hf_push(path, f'step1-shards/{name}', f'shard {name}: {len(qs):,} questions')
        q_mark = len(qs)
        el = (time.time() - t0) / 60
        print(f'  {len(qs):,}/{N:,}  {len(docs):,} pages  {el:.1f} min  '
              f'~{el / len(qs) * (N - len(qs)):.0f} min left  -> {name}')
    if len(qs) >= N:
        break

In [ ]:
# collected, never asserted - a 30-60 min stream must not die on something inspectable
hashes_per_url = defaultdict(set)
for q in qs:
    if q['url']:
        hashes_per_url[q['url']].add(q['doc_hash'])
split  = {u: hs for u, hs in hashes_per_url.items() if len(hs) > 1}   # one url, >1 content
merged = [d['urls'] for d in docs.values() if len(d['urls']) > 1]     # one page, >1 url

# full snapshot
pickle.dump({'documents': docs, 'questions': qs, 'n_streamed': n,
             'dataset_revision': REV, 'max_answer_tokens': MAXT}, open(CKPT, 'wb'))

In [ ]:
# small parquet for studying cleaning locally: tokens + is_html, no 1.4 GB download
samp = list(docs.items())[:SAMPN]
keep = {h for h, _ in samp}
DOCP = str(ART / 'docs_sample.parquet')
QP   = str(ART / 'questions_sample.parquet')
pq.write_table(pa.table({'doc_hash': [h for h, _ in samp],
                         'title'   : [d['title'] for _, d in samp],
                         'urls'    : [d['urls'] for _, d in samp],
                         'token'   : [d['token'] for _, d in samp],
                         'is_html' : [d['is_html'] for _, d in samp]}), DOCP)
sq = [q for q in qs if q['doc_hash'] in keep]
pq.write_table(pa.table({'question'        : [q['question'] for q in sq],
                         'answer'          : [q['answer'] for q in sq],
                         'answers_rendered': [q['answers_rendered'] for q in sq],
                         'doc_hash'        : [q['doc_hash'] for q in sq],
                         'raw_spans'       : [[s for sp in q['raw_spans'] for s in sp]
                                              for q in sq]}), QP)

mins = (time.time() - t0) / 60
print(f'{len(qs):,} questions / {len(docs):,} pages from {n:,} examples in {mins:.1f} min')
print(f'urls with >1 content hash : {len(split)}   <- url-keying would have corrupted these')
print(f'pages merged from >1 url  : {len(merged)}')
print(f'-> {CKPT}  ({os.path.getsize(CKPT) / 1e6:,.0f} MB)')
print(f'-> {DOCP}  ({os.path.getsize(DOCP) / 1e6:,.1f} MB, {len(samp)} docs)')
print(f'-> {QP}  ({os.path.getsize(QP) / 1e6:,.1f} MB, {len(sq)} questions)')

In [ ]:
ck_commit = nqrun.hf_push(CKPT, 'step1-checkpoint-v2/step1_checkpoint.pkl',
                          f'{len(qs):,} questions, NQ {REV[:8]}')
nqrun.hf_push(DOCP, 'step1-sample/docs_sample.parquet', f'{len(samp)} docs for local study')
nqrun.hf_push(QP, 'step1-sample/questions_sample.parquet', f'{len(sq)} questions')
nqrun.log_metrics('1_stream', n_questions=len(qs), n_documents=len(docs), n_streamed=n,
                  urls_split=len(split), pages_merged=len(merged), shards=len(qs) // SHARD,
                  hf_commit=ck_commit, dataset_revision=REV, max_answer_tokens=MAXT,
                  stream_minutes=round(mins, 1))